# Fine-tune PP-OCRv4 rec head on HVAC tag bubbles

Run on Kaggle T4 (or any single-GPU box, ~30-60 min). Input: `ocr_finetune/` produced locally by `prepare_ocr_finetune.py`. Output: `models/rec_ppocr_v4_hvac/{inference.pdmodel,inference.pdiparams,dict.txt}` ready to drop into the repo.

**Train/inference parity:** the dataset was produced with `ocr_preprocess.preprocess_bubble_crop` (3x upscale + Otsu). `tag_matcher._ocr_bubble_crops_paddle` applies the same preprocessing. Do NOT diverge.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Install PaddlePaddle GPU + PaddleOCR (training repo)

In [ ]:
%pip install -q paddlepaddle-gpu==2.5.2 -i https://mirror.baidu.com/pypi/simple
%pip install -q paddleocr==2.7.3 lmdb

In [ ]:
import os, subprocess
if not os.path.exists('PaddleOCR'):
    !git clone -b release/2.7 --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

## 2. Upload the prepared dataset
Upload `ocr_finetune.zip` produced by `prepare_ocr_finetune.py` then unzip.

In [ ]:
# Kaggle auto-extracts uploaded datasets, so /kaggle/input/<slug>/ already
# contains the ocr_finetune/ tree (no .zip to unpack). Locally / Colab the
# upload is the raw .zip; handle both.
import os, glob, shutil
from pathlib import Path

# Case 1: Kaggle dataset attached → look for the train.txt fingerprint
train_txts = glob.glob('/kaggle/input/**/train.txt', recursive=True)
if train_txts:
    src_dir = str(Path(train_txts[0]).parent)
    print('found prepared dataset at', src_dir)
    if not os.path.exists('ocr_finetune'):
        os.symlink(src_dir, 'ocr_finetune')
else:
    # Case 2: raw .zip uploaded
    zips = (glob.glob('/kaggle/input/**/ocr_finetune.zip', recursive=True)
            + glob.glob('/content/ocr_finetune.zip')
            + glob.glob('./ocr_finetune.zip'))
    assert zips, 'ocr_finetune dataset not found — attach it to the notebook'
    print('unpacking', zips[0])
    shutil.unpack_archive(zips[0], '.')

!ls ocr_finetune | head
!wc -l ocr_finetune/train.txt ocr_finetune/val.txt ocr_finetune/dict.txt


## 3. Download PP-OCRv4 English rec pretrained weights

In [ ]:
!mkdir -p pretrain_models
!wget -nc -q https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar -O pretrain_models/en_PP-OCRv4_rec_train.tar
!tar -xf pretrain_models/en_PP-OCRv4_rec_train.tar -C pretrain_models/
!ls pretrain_models/en_PP-OCRv4_rec_train

## 4. Write the fine-tune config

In [ ]:
import yaml

# Load the known-good upstream PP-OCRv4 English rec recipe and override only
# what we need. Hand-rolling the transforms list is fragile (RecConAug needs
# get_ext_data, KeepKeys needs valid_ratio from a specific resize op, etc).
with open('configs/rec/PP-OCRv4/en_PP-OCRv4_rec.yml') as f:
    cfg = yaml.safe_load(f)

cfg['Global']['pretrained_model'] = './pretrain_models/en_PP-OCRv4_rec_train/best_accuracy'
cfg['Global']['character_dict_path'] = './ocr_finetune/dict.txt'
cfg['Global']['max_text_length'] = 12
cfg['Global']['save_model_dir'] = './output/rec_ppocr_v4_hvac/'
cfg['Global']['epoch_num'] = 80
cfg['Global']['use_space_char'] = False
cfg['Global']['save_inference_dir'] = './inference/rec_ppocr_v4_hvac'

cfg['Optimizer']['lr']['learning_rate'] = 0.0005

cfg['Train']['dataset']['data_dir'] = './ocr_finetune/'
cfg['Train']['dataset']['label_file_list'] = ['./ocr_finetune/train.txt']
cfg['Train']['loader']['batch_size_per_card'] = 128
cfg['Train']['loader']['num_workers'] = 4

cfg['Eval']['dataset']['data_dir'] = './ocr_finetune/'
cfg['Eval']['dataset']['label_file_list'] = ['./ocr_finetune/val.txt']
cfg['Eval']['loader']['batch_size_per_card'] = 128
cfg['Eval']['loader']['num_workers'] = 2

out_path = 'configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml'
with open(out_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('train transforms:', [list(t.keys())[0] for t in cfg['Train']['dataset']['transforms']])
print('eval  transforms:', [list(t.keys())[0] for t in cfg['Eval']['dataset']['transforms']])
print('wrote', out_path)


## 5. Train
Expected: 30-60 min on T4. Watch the `acc` metric â€” target â‰¥ 0.95 by epoch 30.

In [ ]:
!python tools/train.py -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml

## 6. Export inference model

In [ ]:
!python tools/export_model.py \
    -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec_hvac.yml \
    -o Global.pretrained_model=./output/rec_ppocr_v4_hvac/best_accuracy \
       Global.save_inference_dir=./inference/rec_ppocr_v4_hvac

In [ ]:
import shutil
shutil.copy('./ocr_finetune/dict.txt', './inference/rec_ppocr_v4_hvac/dict.txt')
shutil.make_archive('/kaggle/working/rec_ppocr_v4_hvac', 'zip', './inference', 'rec_ppocr_v4_hvac')
print('Download /kaggle/working/rec_ppocr_v4_hvac.zip, unzip into models/ in the repo.')